# Temporal confirmation on 19 later die-casting dates

The full competition file extends the initially evaluated training mirror. This check asks whether the inspection rankings from the original 2,048-shot fit retain their yield on all complete observed dates strictly after the initial evaluation. The internal protocol was locked after the earlier results were known and before examining later labels or scores. Both periods come from the same die-casting release. No additional training shots enter the models.

Run this notebook by itself or after the initial validation notebook. It verifies each source against a pinned Git blob. Supply local paths below, or leave them blank to download the pinned files. The published run used Python 3.14.0, NumPy 2.4.3, pandas 3.0.1, SciPy 1.16.3, scikit-learn 1.8.0 and threadpoolctl 3.6.0. Results are written beneath a relative directory that you may change. No raw industrial records are packaged.

In [1]:
from pathlib import Path
from urllib.parse import quote
import os
initial_file = os.environ.get('DIECAST_CSV') or None  # Optional local training-mirror CSV
full_file = os.environ.get('DIECAST_FULL_CSV') or None  # Optional local full-source CSV
initial_output = Path(os.environ.get('DIECAST_RESULTS', 'diecasting_results'))
later_output = initial_output / 'follow_on'
later_output.mkdir(parents=True, exist_ok=True)

## Reconstruct the original inspection model

The functions below fit the five original ranking rules, using the same transformations, four-component representation, feature maps and class weights as the initial die-casting evaluation. The notebook can regenerate the initial score file if it is missing.

In [2]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import platform
from io import BytesIO
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import scipy
from scipy.linalg import hadamard
import sklearn
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits


COMMIT = "3010621441321d2d51f8043d0edc4ac6440cd80f"
BLOB = "5319bc94337c0ad0284553f2611f5ec31c13fd98"
FILE_SIZE = 17_920_698
URL = f"https://raw.githubusercontent.com/yeonduhaeyo/diecasting-project/{COMMIT}/data/train.csv"
PROTOCOL_SHA = "51bdf7844e08ef726cf3beefe358c14c380157777488688ac5611a5ec0231849"
SENSORS = ["molten_temp", "sleeve_temperature", "low_section_speed", "high_section_speed",
           "cast_pressure", "biscuit_thickness", "physical_strength",
           "upper_mold_temp1", "upper_mold_temp2", "upper_mold_temp3",
           "lower_mold_temp1", "lower_mold_temp2", "lower_mold_temp3", "Coolant_temperature",
           "EMS_operation_time", "facility_operation_cycleTime", "production_cycletime"]
MODELS = ["Q4 + RZZ","PCA4 RBF","Q4 no RZZ","PCA4 logistic","Chronology only"]
POLICIES = ["Largest remainder","Cumulative volume"]


def digest(data):
    return hashlib.sha256(data).hexdigest()


def bytes_for_analysis(file):
    if file:
        raw = Path(file).read_bytes()
    else:
        request = Request(URL, headers={"User-Agent":"QML-inspection-reproduction"})
        with urlopen(request, timeout=120) as response:
            raw = response.read()
    git_hash = hashlib.sha1(f"blob {len(raw)}\0".encode("ascii") + raw).hexdigest()
    if len(raw) != FILE_SIZE or git_hash != BLOB:
        raise ValueError("The complete source does not match its fixed byte length and Git blob.")
    return raw


def write_json(path, vals):
    path.write_text(json.dumps(vals, ensure_ascii=False, indent=2, allow_nan=False) + "\n",encoding="utf-8")


def schema(raw):
    for encoding in ("utf-8-sig","cp949"):
        try:
            raw.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError("The CSV could not be decoded under UTF-8 or CP949.")
    columns = pd.read_csv(BytesIO(raw),encoding=encoding,nrows=0).columns.to_list()
    simple = {c.casefold():c for c in columns}
    sensors = []
    for sensor in SENSORS:
        variants = [sensor.casefold()]
        if "mold_temp" in sensor:
            variants.append(sensor.replace("mold_temp","mold_temperature").casefold())
        matches = [simple[v] for v in variants if v in simple]
        if len(matches)>1:
            raise ValueError(f"Ambiguous column for {sensor}: {matches}")
        sensors.extend(matches)
    if len(sensors)<4:
        raise ValueError("Fewer than four eligible source sensor columns.")
    return encoding,sensors,simple["registration_time"],simple["passorfail"],simple.get("mold_code")


def targets(raw, encoding, target):
    values = pd.read_csv(BytesIO(raw), encoding=encoding, usecols=[target],dtype=str)[target]
    return pd.to_numeric(values,errors="coerce")


def four_qubit_states(components, coupled):
    z = 1-2*((np.arange(16)[:,None] >> np.arange(3,-1,-1)) & 1)
    phase = components @ z.T
    if coupled:
        phase += (components[:,:-1]*components[:,1:]) @ (z[:,:-1]*z[:,1:]).T
    diagonal = np.exp(-0.5j*phase)
    states = ((diagonal/4.0) @ (hadamard(16)/4.0)) * diagonal
    np.testing.assert_allclose(np.linalg.norm(states,axis=1),1.0,atol=1e-12)
    return states


def gram(left,right):
    values = np.empty((len(left),len(right)),dtype=np.float64)
    adjoint = right.conj().T
    for start in range(0,len(left),192):
        block = slice(start,start+192)
        values[block] = np.abs(left[block] @ adjoint)**2
    return values


def fit_and_score(train_x,train_y,test_x):
    transform = make_pipeline(SimpleImputer(strategy="median"),VarianceThreshold(),StandardScaler())
    xt = transform.fit_transform(train_x)
    xv = transform.transform(test_x)
    if xt.shape[1]<4:
        raise ValueError("Fewer than four variable numeric training features.")
    pca = PCA(n_components=4,svd_solver="full")
    pc = pca.fit_transform(xt)
    pv = pca.transform(xv)
    gamma = 1/(pc.shape[1]*pc.var())
    rbf = SVC(C=1,class_weight="balanced",kernel="rbf",gamma=gamma).fit(pc,train_y)
    logistic = LogisticRegression(C=1,class_weight="balanced",solver="lbfgs",max_iter=2000,tol=1e-8).fit(pc,train_y)
    scores = {"PCA4 RBF":rbf.decision_function(pv),"PCA4 logistic":logistic.decision_function(pv)}
    scale = 0.5/np.sqrt(np.mean(pc**2))
    for name, coupled in (("Q4 + RZZ",True),("Q4 no RZZ",False)):
        training = four_qubit_states(pc*scale,coupled)
        testing = four_qubit_states(pv*scale,coupled)
        matrix = gram(training,training)
        np.testing.assert_allclose(matrix.diagonal(),1.0,atol=1e-12)
        model = SVC(C=1,kernel="precomputed",class_weight="balanced").fit(matrix,train_y)
        del matrix
        scores[name]=np.concatenate([model.decision_function(gram(testing[i:i+192],training))
                                     for i in range(0,len(testing),192)])
    scores["Chronology only"] = -np.arange(len(pv),dtype=float)
    for score in scores.values():
        if not np.isfinite(score).all():
            raise ValueError("Nonfinite decision scores.")
    retained = transform.get_feature_names_out(train_x.columns).tolist()
    return scores, {"retained_sensors":retained,"pca_variance":float(pca.explained_variance_ratio_.sum()),
                    "quantum_phase_scale":float(scale),"rbf_gamma":float(gamma)}


def quotas(sizes, policy):
    budget=(int(sizes.sum())+9)//10
    if policy=="Largest remainder":
        q,r=np.divmod(budget*sizes,int(sizes.sum()))
        q[np.argsort(-r,kind="stable")[:budget-int(q.sum())]]+=1
    else:
        q=np.diff((np.cumsum(sizes)+9)//10,prepend=0)
    assert int(q.sum())==budget and ((q>=0)&(q<=sizes)).all()
    return q


def freeze(raw, output):
    encoding,sensors,date_col,target,mold = schema(raw)
    data = pd.read_csv(BytesIO(raw),encoding=encoding,usecols=[date_col]+sensors+([mold] if mold else []),dtype=str)
    time = pd.to_datetime(data[date_col],format="mixed",errors="coerce")
    # This is solely a completeness predicate; no held-out label value enters the model or ranking.
    valid_label = targets(raw,encoding,target).isin([0,1])
    eligible = time.notna() & valid_label
    days = pd.Index(time[eligible].dt.normalize().unique()).sort_values()
    cut = 7*len(days)//10
    if min(cut,len(days)-cut)<2:
        raise ValueError("Insufficient distinct recorded dates.")
    order = time[eligible].sort_values(kind="stable").index
    train = order[time[order].dt.normalize()<days[cut]]
    test = order[time[order].dt.normalize()>=days[cut]]
    train_base_size=len(train)
    if len(train)>2048:
        smallest=set(sorted(train,key=lambda i:hashlib.sha256(str(i).encode("ascii")).digest())[:2048])
        train = pd.Index([i for i in train if i in smallest])
    ytrain = targets(raw,encoding,target).loc[train].to_numpy(dtype=np.int8)
    if set(ytrain)!={0,1}:
        raise ValueError("Hash-selected training rows do not contain both classes.")
    x = data[sensors].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
    with threadpool_limits(limits=1):
        scores,model_metadata = fit_and_score(x.loc[train],ytrain,x.loc[test])
    scored = pd.DataFrame({"source_row":test.to_numpy(),"date":time[test].dt.strftime("%Y-%m-%d").to_numpy()})
    for name in MODELS:
        scored[name]=scores[name]
    # The file has no outcomes; float64 scores survive exact roundtrip in pandas.
    scored.to_csv(output/"frozen_scores.csv",index=False,float_format="%.17g",lineterminator="\n")
    group_rows=[g.index.to_numpy() for _,g in scored.groupby("date",sort=True)]
    sizes=np.array([len(r) for r in group_rows],dtype=int)
    chosen=[]
    for policy in POLICIES:
        for rows,q in zip(group_rows,quotas(sizes,policy)):
            for model in MODELS:
                ranks=np.argsort(-scored[model].to_numpy()[rows],kind="stable")
                picks=rows[ranks[:q]]
                chosen.extend((policy,model,scored.date.iloc[r],int(scored.source_row.iloc[r])) for r in picks)
    pd.DataFrame(chosen,columns=["policy","model","date","source_row"]).to_csv(output/"frozen_inspections.csv",index=False,lineterminator="\n")
    meta = {"source_sha256":digest(raw),"git_blob":BLOB,"commit":COMMIT,"protocol_sha256":PROTOCOL_SHA,
            "source_bytes":len(raw),"source_rows":len(data),"timestamp_field":date_col,"label_field":target,
            "invalid_time_rows":int(time.isna().sum()),"invalid_target_rows":int((~valid_label).sum()),
            "excluded_union":int((~eligible).sum()),"eligible_rows":int(eligible.sum()),
            "encoding":encoding,"train_start":str(days[0].date()),"train_end":str(days[cut-1].date()),
            "test_start":str(days[cut].date()),"test_end":str(days[-1].date()),"train_days":cut,"test_days":len(days)-cut,
            "eligible_train":train_base_size,"sampled_train":len(train),"sampled_train_defects":int(ytrain.sum()),
            "test_units":len(test),"inspection_slots":int((len(test)+9)//10),"input_sensors":sensors,
            **model_metadata, "software":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
            "scipy":scipy.__version__,"sklearn":sklearn.__version__},"sampled_train_ids_sha256":digest(np.array(train,dtype="<i8").tobytes()),
            "test_source_rows_sha256":digest(np.array(test,dtype="<i8").tobytes()),
            "scores_sha256":digest((output/"frozen_scores.csv").read_bytes()),
            "selections_sha256":digest((output/"frozen_inspections.csv").read_bytes())}
    if mold:
        def mold_counts(rows):
            return {str(k):int(v) for k,v in data.loc[rows,mold].fillna("(missing)").value_counts().items()}
        meta["mold_code_counts"]={"training_eligible":mold_counts(order[time[order].dt.normalize()<days[cut]]),"sampled_training":mold_counts(train),"test":mold_counts(test)}
    write_json(output/"external_freeze.json",meta)
    print("Frozen",len(test),"test scores per model over",len(days)-cut,"dates.",meta["scores_sha256"])


def evaluate(raw,output,protocol_sha=PROTOCOL_SHA):
    manifest = json.loads((output/"external_freeze.json").read_text(encoding="utf-8"))
    assert manifest["source_sha256"] == digest(raw) and manifest["protocol_sha256"] == protocol_sha
    for f,k in (("frozen_scores.csv","scores_sha256"),("frozen_inspections.csv","selections_sha256")):
        assert digest((output/f).read_bytes()) == manifest[k]
    score = pd.read_csv(output/"frozen_scores.csv",float_precision="round_trip")
    selections=pd.read_csv(output/"frozen_inspections.csv")
    y=targets(raw,manifest["encoding"],manifest["label_field"]).loc[score.source_row].to_numpy(dtype=np.int8)
    assert np.isin(y,[0,1]).all()
    day_indices=[(day,group.index.to_numpy()) for day,group in score.groupby("date",sort=True)]
    byday=[]
    for policy in POLICIES:
        for (day,rows),q in zip(day_indices,quotas(np.array([len(rows) for _,rows in day_indices]),policy)):
            for name in MODELS:
                s=score[name].to_numpy()[rows]
                order=np.argsort(-s,kind="stable")
                mask=(selections.policy==policy)&(selections.model==name)&(selections.date==day)
                assert score.source_row.to_numpy()[rows[order[:q]]].tolist()==selections.loc[mask,"source_row"].to_list()
                labels=y[rows]
                hits=int(labels[order[:q]].sum())
                tie_min=tie_max=tie_expected=0
                if q:
                    boundary=s[order[q-1]]
                    tied=s==boundary
                    above=s>boundary
                    fixed=int(labels[above].sum())
                    nt=int(tied.sum())
                    ntpos=int(labels[tied].sum())
                    free=int(q-above.sum())
                    tie_min=fixed+max(0,free-(nt-ntpos))
                    tie_max=fixed+min(free,ntpos)
                    tie_expected=fixed+free*ntpos/nt
                assert tie_min<=hits<=tie_max
                byday.append((policy,name,day,len(rows),int(q),int(labels.sum()),hits,tie_min,tie_max,tie_expected))
    daily=pd.DataFrame(byday, columns=["policy","model","date","units","slots","defects","hits","tie_min","tie_max","tie_expected"])
    daily.to_csv(output/"external_by_date.csv",index=False)
    result=[]
    for (policy,name),g in daily.groupby(["policy","model"],sort=False):
        pos=int(y.sum())
        total=int(g.hits.sum())
        result.append({"policy":policy,"model":name,"units":len(y),"defects":pos,"slots":int(g.slots.sum()),
                       "hits":total,"recall":total/pos if pos else np.nan,
                       "test_dates":len(g),"positive_dates":int((g.defects>0).sum()),"dates_reached":int((g.hits>0).sum()),
                       "tie_min":int(g.tie_min.sum()),"tie_max":int(g.tie_max.sum()),"tie_expected":g.tie_expected.sum(),
                       "pooled_ap":average_precision_score(y,score[name]) if pos else np.nan,
                       "uniform_expected":float((g.slots*g.defects/g.units).sum()),"daily_ceiling":int(np.minimum(g.slots,g.defects).sum())})
    totals=pd.DataFrame(result)
    totals.to_csv(output/"external_summary.csv",index=False)
    primary=daily.loc[daily.policy.eq("Largest remainder")]
    q4=primary.loc[primary.model.eq("Q4 + RZZ")].set_index("date").sort_index()
    rbf=primary.loc[primary.model.eq("PCA4 RBF")].set_index("date").loc[q4.index]
    uncoupled=primary.loc[primary.model.eq("Q4 no RZZ")].set_index("date").loc[q4.index]
    differences=(q4.hits-rbf.hits).to_numpy(dtype=int)
    slots=q4.slots.to_numpy(dtype=int)
    sensitivity={"unit":"observed date","difference_order":"coupled minus matched RBF",
                 "observed_detection_difference":int(differences.sum()),
                 "coupled_minus_uncoupled":int((q4.hits-uncoupled.hits).sum()),
                 "observed_difference_per_100_inspections":float(100*differences.sum()/slots.sum()),
                 "coupled_wins":int((differences>0).sum()),"ties":int((differences==0).sum()),
                 "coupled_losses":int((differences<0).sum()),
                 "uncertainty_note_sha256":"e5069e44ef1d23b30d416de464c3d15723358dbd8de98db87bdd5ec7ebd981a1"}
    if len(slots)>=15:
        n=len(slots)
        starts=np.random.default_rng(20261123).integers(0,n,size=(10000,(n+4)//5))
        samples=((starts[:,:,None]+np.arange(5))%n).reshape(10000,-1)[:,:n]
        den=slots[samples].sum(axis=1)
        stats=100*differences[samples].sum(axis=1)[den>0]/den[den>0]
        lo,hi=np.percentile(stats,[2.5,97.5])
        sensitivity.update(block_length=5,bootstrap_seed=20261123,replicates=10000,
                           usable_replicates=int(len(stats)),lower_percentile=float(lo),upper_percentile=float(hi))
    write_json(output/"external_primary_contrast.json",sensitivity)
    print(totals.to_string(index=False))
    print(json.dumps(sensitivity,indent=2))

## Verify the common prefix and freeze later scores

The two files agree in their sensor fields and registration times on the first 73,612 records. Of these, 73,611 have observed binary targets in both files, all concordant. Zero-based row 19,327, recorded on 25 January 2019 at 19:09:09, has a blank target in the full release and a zero in the mirror. Its origin is undocumented. This row contributed to neither model fitting nor either evaluation. The check below verifies the discrepancy and its exclusion.

The same in-memory model objects must reproduce every initial score on 26,916 shots exactly before they are used to rank the later shots. March 12 is excluded in its entirety from the follow-on. Candidate scores are frozen and hashed for all shots on March 13–31 before using the later target values, including to determine eligibility.

In [3]:
FOLLOW_COMMIT="ad2b1147ea73fcdd54fed0507ed2f73da3c652b9"
FOLLOW_BLOB="fa4909461f32ee163686b04ac51e7c9f95f6b753"
FOLLOW_SHA="1894052d1091642a110cd76710e9d8bd995d4a30188a75d6c4fe184bf82dcf4f"
FOLLOW_PROTOCOL="2276c711d441a4c2986f9604f94a682f6070d874d8180a530eef961a571389f2"
FULL_URL=f"https://raw.githubusercontent.com/greatsangho/2024-Kamp-Competition/{FOLLOW_COMMIT}/"+quote("data/경진대회용 주조 공정최적화 데이터셋.csv",safe="/")


def full_source(file):
    if file:
        data=Path(file).read_bytes()
    else:
        with urlopen(Request(FULL_URL,headers={"User-Agent":"QML-inspection-reproduction"}),timeout=120) as response:
            data=response.read()
    git_sha=hashlib.sha1(f"blob {len(data)}\0".encode("ascii")+data).hexdigest()
    if len(data)!=21479807 or digest(data)!=FOLLOW_SHA or git_sha!=FOLLOW_BLOB:
        raise ValueError("Full-source file identity mismatch.")
    return data


def rebuild_predict(train_x,y):
    process=make_pipeline(SimpleImputer(strategy="median"),VarianceThreshold(),StandardScaler())
    tx=process.fit_transform(train_x)
    pca=PCA(n_components=4,svd_solver="full")
    pc=pca.fit_transform(tx)
    scale=0.5/np.sqrt(np.mean(pc**2))
    classical=SVC(C=1,class_weight="balanced",kernel="rbf",gamma=1/(pc.shape[1]*pc.var())).fit(pc,y)
    linear=LogisticRegression(C=1,class_weight="balanced",solver="lbfgs",max_iter=2000,tol=1e-8).fit(pc,y)
    kernels={}
    for name,coupled in (("Q4 + RZZ",True),("Q4 no RZZ",False)):
        train_states=four_qubit_states(pc*scale,coupled)
        model=SVC(C=1,kernel="precomputed",class_weight="balanced").fit(gram(train_states,train_states),y)
        kernels[name]=(coupled,train_states,model)
    def predict(x):
        pv=pca.transform(process.transform(x))
        scores={"PCA4 RBF":classical.decision_function(pv),"PCA4 logistic":linear.decision_function(pv),
                "Chronology only":-np.arange(len(pv),dtype=float)}
        for name,(coupled,train_states,model) in kernels.items():
            sv=four_qubit_states(pv*scale,coupled)
            scores[name]=np.concatenate([model.decision_function(gram(sv[i:i+192],train_states)) for i in range(0,len(sv),192)])
        return scores
    return predict


def freeze_follow_on(original,full,first_output,out):
    first=json.loads((first_output/"external_freeze.json").read_text(encoding="utf-8"))
    enc,sensors,timecol,label,mold=schema(original)
    ec,s2,t2,l2,m2=schema(full)
    assert (sensors,timecol,label,mold)==(s2,t2,l2,m2) and sensors==first["input_sensors"]
    historic_y=targets(original,enc,label)
    # Read only previously evaluated label values from the longer source.
    old_from_full=pd.to_numeric(pd.read_csv(BytesIO(full),encoding=ec,usecols=[label],nrows=len(historic_y),dtype=str)[label],errors="coerce")
    joint=historic_y.isin([0,1])&old_from_full.isin([0,1])
    np.testing.assert_array_equal(historic_y[joint].to_numpy(),old_from_full[joint].to_numpy())
    unpaired=historic_y.index[~joint].tolist()
    assert unpaired==[19327] and historic_y.loc[19327]==0 and pd.isna(old_from_full.loc[19327])
    a=pd.read_csv(BytesIO(original),encoding=enc,usecols=[timecol]+sensors,dtype=str)
    b=pd.read_csv(BytesIO(full),encoding=ec,usecols=[timecol]+sensors,dtype=str)
    pd.testing.assert_frame_equal(a,b.iloc[:len(a)].copy())
    times=pd.to_datetime(a[timecol],format="mixed",errors="coerce")
    eligible=times.notna()&historic_y.isin([0,1])
    order=times[eligible].sort_values(kind="stable").index
    earlier=order[times[order].dt.normalize()<=pd.Timestamp(first["train_end"])]
    ids=set(sorted(earlier,key=lambda x:hashlib.sha256(str(x).encode("ascii")).digest())[:2048])
    chosen=pd.Index([idx for idx in earlier if idx in ids])
    assert digest(np.asarray(chosen,dtype="<i8").tobytes())==first["sampled_train_ids_sha256"]
    historical=pd.read_csv(first_output/"frozen_scores.csv",float_precision="round_trip")
    assert not set(unpaired).intersection(chosen) and not set(unpaired).intersection(historical.source_row)
    assert digest((first_output/"frozen_scores.csv").read_bytes())==first["scores_sha256"]
    full_times=pd.to_datetime(b[timecol],format="mixed",errors="coerce")
    later=full_times[full_times.dt.normalize()>pd.Timestamp(first["test_end"])].sort_values(kind="stable").index
    x=a[sensors].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
    new_x=b.loc[later,sensors].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
    with threadpool_limits(limits=1):
        model=rebuild_predict(x.loc[chosen],historic_y.loc[chosen].to_numpy(dtype=np.int8))
        reproduced=model(x.loc[historical.source_row])
        for name in MODELS:
            np.testing.assert_array_equal(reproduced[name],historical[name].to_numpy())
        later_scores=model(new_x)
    scored=pd.DataFrame({"source_row":later.to_numpy(),"date":full_times[later].dt.strftime("%Y-%m-%d").to_numpy()})
    for name in MODELS:
        scored[name]=later_scores[name]
        assert np.isfinite(scored[name]).all()
    scored.to_csv(out/"frozen_candidate_scores.csv",index=False,float_format="%.17g",lineterminator="\n")
    info={"protocol_sha256":FOLLOW_PROTOCOL,"source_sha256":digest(full),"git_blob":FOLLOW_BLOB,
          "source_rows":len(b),"old_source_rows":len(a),"raw_candidate_rows":len(scored),"candidate_dates":scored.date.nunique(),
          "start":scored.date.min(),"end":scored.date.max(),"invalid_timestamps_full_file":int(full_times.isna().sum()),
          "last_date_suffix_rows_excluded":int(sum(full_times.iloc[len(a):].dt.normalize()==pd.Timestamp(first["test_end"]))),
          "prefix_labels_checked":len(historic_y),"prefix_binary_pairs_matching":int(joint.sum()),
          "prefix_binary_pair_conflicts":0,"prefix_full_file_missing_target_rows":unpaired,
          "unpaired_target":{"source_row":19327,"registration_time":times.loc[19327].isoformat(),
                             "original_mirror_label":0,"full_source_label":None,
                             "used_in_fit":False,"used_in_first_test":False},
          "first_test_score_values_verified":len(historical)*len(MODELS),
          "sampled_train_ids_sha256":first["sampled_train_ids_sha256"],"previous_score_digest":first["scores_sha256"],
          "frozen_candidate_score_sha256":digest((out/"frozen_candidate_scores.csv").read_bytes()),
          "retained_sensors":first["retained_sensors"],"encoding":ec,"label_field":l2,"matched_reference_scores":"bit-for-bit"}
    write_json(out/"followon_candidate_freeze.json",info)
    print("New period candidate scores frozen:",info["raw_candidate_rows"],"rows,",info["candidate_dates"],"dates,",info["frozen_candidate_score_sha256"])

In [4]:
initial_raw = bytes_for_analysis(initial_file)
extended_raw = full_source(full_file)
if not all((initial_output / name).exists() for name in ('external_freeze.json', 'frozen_scores.csv')):
    freeze(initial_raw, initial_output)
freeze_follow_on(initial_raw, extended_raw, initial_output, later_output)
candidate_manifest = json.loads((later_output / 'followon_candidate_freeze.json').read_text(encoding='utf-8'))
print('Reproduced initial score values:', candidate_manifest['first_test_score_values_verified'])
print('Excluded partial last-date suffix:', candidate_manifest['last_date_suffix_rows_excluded'])

Frozen 26916 test scores per model over 20 dates. 97f4f08cc453d432723846867462839e8d46145cb6f9d315249f6db52554ea22
New period candidate scores frozen: 17306 rows, 19 dates, 9fbee254f76c59599fdb13403f608d2ca213b4cf8361d0bf9cb619e4f9d896de
Reproduced initial score values: 134580
Excluded partial last-date suffix: 1097


## Link released outcomes and evaluate the fixed rankings

Binary outcome availability defines the eligible inspection pool. The primary allocation takes the largest remainders at 10% total capacity and breaks within-day score ties chronologically. Daily selections, defect yield, event coverage, average precision and the secondary running-allocation totals are recorded. The circular five-observed-day bootstrap resamples daily differences and capacities with the fitted scores and quotas fixed. Its interval describes this single production period.

In [5]:
def evaluate_follow_on(full,out):
    info=json.loads((out/"followon_candidate_freeze.json").read_text(encoding="utf-8"))
    assert info["source_sha256"]==digest(full) and info["frozen_candidate_score_sha256"]==digest((out/"frozen_candidate_scores.csv").read_bytes())
    scored=pd.read_csv(out/"frozen_candidate_scores.csv",float_precision="round_trip")
    # Outcome validity determines eligible capacity only after all candidate scores are fixed.
    valid=targets(full,info["encoding"],info["label_field"]).loc[scored.source_row].isin([0,1]).to_numpy()
    eligible=scored.loc[valid].reset_index(drop=True)
    if not len(eligible):
        write_json(out/"followon_no_eligible_outcomes.json",{"eligible_rows":0,"raw_candidate_rows":len(scored),"test_dates_with_scores":scored.date.nunique(),"recall":None})
        print("None of the candidate records has a released binary target; recall is undefined.")
        return
    eligible.to_csv(out/"frozen_scores.csv",index=False,float_format="%.17g",lineterminator="\n")
    groups=[r.index.to_numpy() for _,r in eligible.groupby("date",sort=True)]
    sizes=np.array([len(r) for r in groups],dtype=int)
    picks=[]
    for policy in POLICIES:
        for rows,q in zip(groups,quotas(sizes,policy)):
            for name in MODELS:
                order=np.argsort(-eligible[name].to_numpy()[rows],kind="stable")
                picks.extend((policy,name,eligible.date.iloc[r],int(eligible.source_row.iloc[r])) for r in rows[order[:q]])
    pd.DataFrame(picks,columns=["policy","model","date","source_row"]).to_csv(out/"frozen_inspections.csv",index=False,lineterminator="\n")
    info.update(eligible_rows=len(eligible),eligible_dates=eligible.date.nunique(),removed_invalid_label=int((~valid).sum()),
                scores_sha256=digest((out/"frozen_scores.csv").read_bytes()),selections_sha256=digest((out/"frozen_inspections.csv").read_bytes()))
    write_json(out/"external_freeze.json",info)
    evaluate(full,out,protocol_sha=FOLLOW_PROTOCOL)

In [6]:
evaluate_follow_on(extended_raw, later_output)

           policy           model  units  defects  slots  hits   recall  test_dates  positive_dates  dates_reached  tie_min  tie_max  tie_expected  pooled_ap  uniform_expected  daily_ceiling
Largest remainder        Q4 + RZZ  17306      674   1731   516 0.765579          19              19             19      516      516         516.0   0.629774         67.339770            663
Largest remainder        PCA4 RBF  17306      674   1731   432 0.640950          19              19             19      432      432         432.0   0.252514         67.339770            663
Largest remainder       Q4 no RZZ  17306      674   1731   523 0.775964          19              19             19      523      523         523.0   0.522964         67.339770            663
Largest remainder   PCA4 logistic  17306      674   1731   556 0.824926          19              19             19      556      556         556.0   0.751216         67.339770            663
Largest remainder Chronology only  17306     